In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from imblearn.over_sampling import SMOTENC
from imblearn.under_sampling import RandomUnderSampler

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "simulasi":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

PROCESSED_DIR = PROJECT_ROOT / "dataset" / "processed"
SPLIT_DIR = PROJECT_ROOT / "output" / "01_split_validation"
OUTPUT_DIR = PROJECT_ROOT / "output" / "02_preprocessing_scenario"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT ROOT :", PROJECT_ROOT)
print("OUTPUT       :", OUTPUT_DIR)

PROJECT ROOT : /Users/riskyduha/Documents/Tesis/Prediction_Non_Adherent
OUTPUT       : /Users/riskyduha/Documents/Tesis/Prediction_Non_Adherent/output/02_preprocessing_scenario


In [3]:
DATA_PATH = PROCESSED_DIR / "cimas_medication_adherence_clean.csv"

df = pd.read_csv(DATA_PATH)

outer_assignment = pd.read_csv(
    SPLIT_DIR / "01_outer_split_assignment.csv"
)

inner_assignment = pd.read_csv(
    SPLIT_DIR / "05_inner_cv_assignments.csv"
)

print("Dataset :", df.shape)
print("Outer   :", outer_assignment.shape)
print("Inner   :", inner_assignment.shape)

Dataset : (8141, 16)
Outer   : (8141, 4)
Inner   : (19536, 4)


In [4]:
if "OBSERVATION_ID" not in df.columns:
    df.insert(
        0,
        "OBSERVATION_ID",
        np.arange(1, len(df) + 1)
    )

check = df[
    ["OBSERVATION_ID", "BENEFICIARY", "FINALTHRESHOLD"]
].merge(
    outer_assignment[
        ["OBSERVATION_ID", "BENEFICIARY", "FINALTHRESHOLD"]
    ],
    on="OBSERVATION_ID",
    suffixes=("_data", "_split")
)

assert (
    check["BENEFICIARY_data"].astype(str)
    == check["BENEFICIARY_split"].astype(str)
).all()

assert (
    check["FINALTHRESHOLD_data"]
    == check["FINALTHRESHOLD_split"]
).all()

print("✓ Dataset sinkron dengan hasil split sebelumnya.")

✓ Dataset sinkron dengan hasil split sebelumnya.


In [5]:
development_ids = outer_assignment.loc[
    outer_assignment["partisi"] == "DATA_PENGEMBANGAN",
    "OBSERVATION_ID"
]

df_dev = df[
    df["OBSERVATION_ID"].isin(development_ids)
].copy()

print("Data Pengembangan:", len(df_dev))

assert len(df_dev) == 6512

Data Pengembangan: 6512


In [ ]:

# AUDIT KETAT: identitas data, assignment, dan isolasi pasien
from collections import Counter

required = {"OBSERVATION_ID", "BENEFICIARY", "FINALTHRESHOLD"}
for name, frame in [("dataset",df), ("outer",outer_assignment), ("inner",inner_assignment)]:
    missing = required.intersection({"OBSERVATION_ID"}) - set(frame.columns)
    assert not missing, f"{name} tanpa kolom {missing}"
    assert frame["OBSERVATION_ID"].notna().all(), f"{name}: OBSERVATION_ID kosong"

assert df["OBSERVATION_ID"].is_unique, "Duplikasi OBSERVATION_ID pada dataset"
assert outer_assignment["OBSERVATION_ID"].is_unique, "Duplikasi outer assignment"
assert set(df["OBSERVATION_ID"]) == set(outer_assignment["OBSERVATION_ID"]), "Outer assignment bukan pemetaan 1:1 seluruh observasi"

dev_ids = set(df_dev["OBSERVATION_ID"])
test_ids = set(outer_assignment.loc[
    outer_assignment["partisi"] == "DATA_UJI", "OBSERVATION_ID"])
assert not dev_ids.intersection(test_ids), "Observasi development dan test overlap"
assert dev_ids | test_ids == set(df["OBSERVATION_ID"]), "Partisi tidak mencakup seluruh observasi"

groups_dev = set(df.loc[df["OBSERVATION_ID"].isin(dev_ids), "BENEFICIARY"].astype(str))
groups_test = set(df.loc[df["OBSERVATION_ID"].isin(test_ids), "BENEFICIARY"].astype(str))
assert not groups_dev.intersection(groups_test), "BENEFICIARY overlap development/test"

assert {"repetition","seed","validation_fold","OBSERVATION_ID"}.issubset(inner_assignment.columns)
assert set(inner_assignment["repetition"]) == {1,2,3}, "Repetisi bukan 1,2,3"
seed_expected = {1:101, 2:202, 3:303}
assignment_audit=[]
for rep, group in inner_assignment.groupby("repetition"):
    assert set(group["OBSERVATION_ID"]) == dev_ids, f"Rep {rep}: ID development tidak lengkap"
    assert group["OBSERVATION_ID"].is_unique, f"Rep {rep}: satu observasi >1 fold validasi"
    assert set(group["validation_fold"]) == {1,2,3,4,5}, f"Rep {rep}: fold tidak lengkap"
    assert set(group["seed"]) == {seed_expected[int(rep)]}, f"Rep {rep}: seed tidak cocok"
    for fold in range(1,6):
        valid = set(group.loc[group["validation_fold"]==fold,"OBSERVATION_ID"])
        train_ids_check = dev_ids - valid
        gtr = set(df.loc[df["OBSERVATION_ID"].isin(train_ids_check),"BENEFICIARY"].astype(str))
        gva = set(df.loc[df["OBSERVATION_ID"].isin(valid),"BENEFICIARY"].astype(str))
        overlap = len(gtr & gva)
        assignment_audit.append({"repetition":int(rep),"seed":seed_expected[int(rep)],
            "fold":fold,"n_train":len(train_ids_check),"n_validation":len(valid),
            "n_group_overlap":overlap,"n_development_test_group_overlap":len(groups_dev & groups_test)})
        assert overlap == 0, f"Rep {rep} fold {fold}: BENEFICIARY overlap={overlap}"

assignment_audit = pd.DataFrame(assignment_audit)
assignment_audit.to_csv(OUTPUT_DIR/"07_group_assignment_checks.csv",index=False)

# Bandingkan distribusi kelas split aktual dengan angka yang dilaporkan
partition_records=[]
for part, ids in [("DATA_PENGEMBANGAN",dev_ids),("DATA_UJI",test_ids)]:
    section=df[df["OBSERVATION_ID"].isin(ids)]
    counts=section["FINALTHRESHOLD"].value_counts()
    partition_records.append({"partisi":part,"observasi":len(section),
         "beneficiary_unik":section["BENEFICIARY"].nunique(),
         "NON_ADHERENT":int(counts.get("NON-ADHERENT",0)),
         "ADHERENT":int(counts.get("ADHERENT",0))})
partition_audit=pd.DataFrame(partition_records)
partition_audit.to_csv(OUTPUT_DIR/"08_holdout_class_audit.csv",index=False)
display(partition_audit,assignment_audit)
print("PASS: outer dan inner assignment, group isolation, serta kelengkapan fold.")


In [6]:
TARGET = "FINALTHRESHOLD"
GROUP = "BENEFICIARY"

NUMERIC_FEATURES = [
    "AGE",
    "ANNUALCONTRIBUTION",
    "ANNUALCLAIMAMOUNT",
    "UNITS"
]

CATEGORICAL_FEATURES = [
    "Diagnosis",
    "SCHEMETYPE",
    "HOSPITALCOVERTYPE",
    "WELLNESSPROGRAM",
    "GENDER",
    "COMPLICATIONDVELOPMENT",
    "COMORBIDITY.1"
]

FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

X_dev = df_dev[
    ["OBSERVATION_ID"] + FEATURES
].copy()

y_dev = df_dev[
    ["OBSERVATION_ID", TARGET]
].copy()

In [7]:
preprocessing_records = []
imputation_records = []
scenario_records = []
weight_records = []

In [8]:
repetitions = sorted(
    inner_assignment["repetition"].unique()
)

for repetition in repetitions:

    rep_assignment = inner_assignment[
        inner_assignment["repetition"] == repetition
    ]

    seed = int(rep_assignment["seed"].iloc[0])

    for fold in sorted(
        rep_assignment["validation_fold"].unique()
    ):

        # --------------------------------------------------
        # Tentukan validation ID dan training ID
        # --------------------------------------------------

        val_ids = set(
            rep_assignment.loc[
                rep_assignment["validation_fold"] == fold,
                "OBSERVATION_ID"
            ]
        )

        dev_ids = set(df_dev["OBSERVATION_ID"])

        train_ids = dev_ids - val_ids

        train = df_dev[
            df_dev["OBSERVATION_ID"].isin(train_ids)
        ].copy()

        val = df_dev[
            df_dev["OBSERVATION_ID"].isin(val_ids)
        ].copy()

        X_train = train[FEATURES].copy()
        y_train = train[TARGET].copy()

        X_val = val[FEATURES].copy()
        y_val = val[TARGET].copy()

        # ==================================================
        # 1. AUDIT BEFORE PREPROCESSING
        # ==================================================

        train_missing_before = int(
            X_train.isna().sum().sum()
        )

        val_missing_before = int(
            X_val.isna().sum().sum()
        )

        train_rows_before = len(X_train)
        val_rows_before = len(X_val)

        # ==================================================
        # 2. CATEGORICAL IMPUTATION — FIT TRAIN ONLY
        # ==================================================

        cat_imputer = SimpleImputer(
            strategy="most_frequent"
        )

        cat_imputer.fit(
            X_train[CATEGORICAL_FEATURES]
        )

        X_train[CATEGORICAL_FEATURES] = (
            cat_imputer.transform(
                X_train[CATEGORICAL_FEATURES]
            )
        )

        X_val[CATEGORICAL_FEATURES] = (
            cat_imputer.transform(
                X_val[CATEGORICAL_FEATURES]
            )
        )

        # Catat mode yang dipelajari untuk setiap fitur
        for col, statistic in zip(
            CATEGORICAL_FEATURES,
            cat_imputer.statistics_
        ):
            imputation_records.append({
                "repetition": repetition,
                "seed": seed,
                "fold": fold,
                "atribut": col,
                "nilai_imputasi_training": statistic
            })

        # ==================================================
        # 3. ROBUST SCALER — FIT TRAIN ONLY
        # ==================================================

        scaler = RobustScaler()

        scaler.fit(
            X_train[NUMERIC_FEATURES]
        )

        X_train[NUMERIC_FEATURES] = (
            scaler.transform(
                X_train[NUMERIC_FEATURES]
            )
        )

        X_val[NUMERIC_FEATURES] = (
            scaler.transform(
                X_val[NUMERIC_FEATURES]
            )
        )

        # ==================================================
        # 4. POST-PREPROCESSING AUDIT
        # ==================================================

        preprocessing_records.append({
            "repetition": repetition,
            "seed": seed,
            "fold": fold,

            "training_rows_before": train_rows_before,
            "training_rows_after": len(X_train),

            "validation_rows_before": val_rows_before,
            "validation_rows_after": len(X_val),

            "training_missing_before": train_missing_before,
            "training_missing_after":
                int(X_train.isna().sum().sum()),

            "validation_missing_before": val_missing_before,
            "validation_missing_after":
                int(X_val.isna().sum().sum())
        })

        # ==================================================
        # Siapkan dataframe mixed-feature untuk treatment
        # ==================================================

        X_train_mixed = X_train.copy()

        # SMOTENC lebih eksplisit jika categorical menjadi category
        for col in CATEGORICAL_FEATURES:
            X_train_mixed[col] = (
                X_train_mixed[col].astype("category")
            )

        original_non = int(
            (y_train == "NON-ADHERENT").sum()
        )

        original_adh = int(
            (y_train == "ADHERENT").sum()
        )

        # ==================================================
        # S0 — BASELINE
        # ==================================================

        scenario_records.append({
            "repetition": repetition,
            "seed": seed,
            "fold": fold,
            "scenario": "S0",
            "method": "Baseline",
            "before_NON_ADHERENT": original_non,
            "before_ADHERENT": original_adh,
            "after_NON_ADHERENT": original_non,
            "after_ADHERENT": original_adh,
            "after_total": len(y_train)
        })

        # ==================================================
        # S1 — SMOTENC
        # ==================================================

        smotenc = SMOTENC(
            categorical_features=CATEGORICAL_FEATURES,
            sampling_strategy=1.0,
            random_state=42
        )

        X_s1, y_s1 = smotenc.fit_resample(
            X_train_mixed,
            y_train
        )

        scenario_records.append({
            "repetition": repetition,
            "seed": seed,
            "fold": fold,
            "scenario": "S1",
            "method": "SMOTENC",
            "before_NON_ADHERENT": original_non,
            "before_ADHERENT": original_adh,
            "after_NON_ADHERENT":
                int((y_s1 == "NON-ADHERENT").sum()),
            "after_ADHERENT":
                int((y_s1 == "ADHERENT").sum()),
            "after_total": len(y_s1)
        })

        # ==================================================
        # S2 — RANDOM UNDERSAMPLING
        # ==================================================

        rus = RandomUnderSampler(
            sampling_strategy=1.0,
            random_state=42
        )

        X_s2, y_s2 = rus.fit_resample(
            X_train,
            y_train
        )

        scenario_records.append({
            "repetition": repetition,
            "seed": seed,
            "fold": fold,
            "scenario": "S2",
            "method": "Random Undersampling",
            "before_NON_ADHERENT": original_non,
            "before_ADHERENT": original_adh,
            "after_NON_ADHERENT":
                int((y_s2 == "NON-ADHERENT").sum()),
            "after_ADHERENT":
                int((y_s2 == "ADHERENT").sum()),
            "after_total": len(y_s2)
        })

        # ==================================================
        # S3 — INVERSE-FREQUENCY WEIGHTING
        # ==================================================

        n = len(y_train)
        k = y_train.nunique()

        class_counts = y_train.value_counts()

        weight_non = (
            n / (k * class_counts["NON-ADHERENT"])
        )

        weight_adh = (
            n / (k * class_counts["ADHERENT"])
        )

        weight_records.append({
            "repetition": repetition,
            "seed": seed,
            "fold": fold,
            "NON_ADHERENT_count": original_non,
            "ADHERENT_count": original_adh,
            "NON_ADHERENT_weight": weight_non,
            "ADHERENT_weight": weight_adh
        })

        scenario_records.append({
            "repetition": repetition,
            "seed": seed,
            "fold": fold,
            "scenario": "S3",
            "method": "Class Weighting",
            "before_NON_ADHERENT": original_non,
            "before_ADHERENT": original_adh,
            "after_NON_ADHERENT": original_non,
            "after_ADHERENT": original_adh,
            "after_total": len(y_train)
        })

In [9]:
preprocessing_audit = pd.DataFrame(
    preprocessing_records
)

display(preprocessing_audit)

preprocessing_audit.to_csv(
    OUTPUT_DIR / "01_preprocessing_fold_audit.csv",
    index=False
)

,repetition,seed,fold,training_rows_before,training_rows_after,validation_rows_before,validation_rows_after,training_missing_before,training_missing_after,validation_missing_before,validation_missing_after
0,1,101,1,5209,5209,1303,1303,3,0,0,0
1,1,101,2,5209,5209,1303,1303,2,0,1,0
2,1,101,3,5210,5210,1302,1302,1,0,2,0
3,1,101,4,5210,5210,1302,1302,3,0,0,0
4,1,101,5,5210,5210,1302,1302,3,0,0,0
5,2,202,1,5209,5209,1303,1303,1,0,2,0
6,2,202,2,5209,5209,1303,1303,3,0,0,0
7,2,202,3,5210,5210,1302,1302,2,0,1,0
8,2,202,4,5210,5210,1302,1302,3,0,0,0
9,2,202,5,5210,5210,1302,1302,3,0,0,0


In [10]:
imputation_summary = pd.DataFrame(
    imputation_records
)

display(
    imputation_summary[
        imputation_summary["atribut"] == "GENDER"
    ]
)

imputation_summary.to_csv(
    OUTPUT_DIR / "02_imputation_summary.csv",
    index=False
)

,repetition,seed,fold,atribut,nilai_imputasi_training
4,1,101,1,GENDER,FEMALE
11,1,101,2,GENDER,FEMALE
18,1,101,3,GENDER,FEMALE
25,1,101,4,GENDER,FEMALE
32,1,101,5,GENDER,FEMALE
39,2,202,1,GENDER,FEMALE
46,2,202,2,GENDER,FEMALE
53,2,202,3,GENDER,FEMALE
60,2,202,4,GENDER,FEMALE
67,2,202,5,GENDER,FEMALE


In [11]:
scenario_distribution = pd.DataFrame(
    scenario_records
)

scenario_distribution["after_ratio_majority_minority"] = (
    scenario_distribution[
        ["after_NON_ADHERENT", "after_ADHERENT"]
    ].max(axis=1)
    /
    scenario_distribution[
        ["after_NON_ADHERENT", "after_ADHERENT"]
    ].min(axis=1)
)

display(scenario_distribution)

scenario_distribution.to_csv(
    OUTPUT_DIR / "03_scenario_class_distribution.csv",
    index=False
)

,repetition,seed,fold,scenario,method,before_NON_ADHERENT,before_ADHERENT,after_NON_ADHERENT,after_ADHERENT,after_total,after_ratio_majority_minority
0,1,101,1,S0,Baseline,4223,986,4223,986,5209,4.282961
1,1,101,1,S1,SMOTENC,4223,986,4223,4223,8446,1.000000
2,1,101,1,S2,Random Undersampling,4223,986,986,986,1972,1.000000
3,1,101,1,S3,Class Weighting,4223,986,4223,986,5209,4.282961
4,1,101,2,S0,Baseline,4223,986,4223,986,5209,4.282961
5,1,101,2,S1,SMOTENC,4223,986,4223,4223,8446,1.000000
6,1,101,2,S2,Random Undersampling,4223,986,986,986,1972,1.000000
7,1,101,2,S3,Class Weighting,4223,986,4223,986,5209,4.282961
8,1,101,3,S0,Baseline,4223,987,4223,987,5210,4.278622
9,1,101,3,S1,SMOTENC,4223,987,4223,4223,8446,1.000000


In [12]:
class_weight_summary = pd.DataFrame(
    weight_records
)

display(class_weight_summary)

class_weight_summary.to_csv(
    OUTPUT_DIR / "04_class_weight_summary.csv",
    index=False
)

,repetition,seed,fold,NON_ADHERENT_count,ADHERENT_count,NON_ADHERENT_weight,ADHERENT_weight
0,1,101,1,4223,986,0.616742,2.641481
1,1,101,2,4223,986,0.616742,2.641481
2,1,101,3,4223,987,0.616860,2.639311
3,1,101,4,4223,987,0.616860,2.639311
4,1,101,5,4224,986,0.616714,2.641988
5,2,202,1,4223,986,0.616742,2.641481
6,2,202,2,4223,986,0.616742,2.641481
7,2,202,3,4223,987,0.616860,2.639311
8,2,202,4,4224,986,0.616714,2.641988
9,2,202,5,4223,987,0.616860,2.639311


In [13]:
scenario_summary = (
    scenario_distribution
    .groupby(["scenario", "method"])
    .agg(
        min_total_after=("after_total", "min"),
        max_total_after=("after_total", "max"),
        min_non_adherent=("after_NON_ADHERENT", "min"),
        max_non_adherent=("after_NON_ADHERENT", "max"),
        min_adherent=("after_ADHERENT", "min"),
        max_adherent=("after_ADHERENT", "max"),
        min_ratio=("after_ratio_majority_minority", "min"),
        max_ratio=("after_ratio_majority_minority", "max")
    )
    .reset_index()
)

display(scenario_summary)

scenario_summary.to_csv(
    OUTPUT_DIR / "05_scenario_summary.csv",
    index=False
)

,scenario,method,min_total_after,max_total_after,min_non_adherent,max_non_adherent,min_adherent,max_adherent,min_ratio,max_ratio
0,S0,Baseline,5209,5210,4223,4224,986,987,4.278622,4.283976
1,S1,SMOTENC,8446,8448,4223,4224,4223,4224,1.000000,1.000000
2,S2,Random Undersampling,1972,1974,986,987,986,987,1.000000,1.000000
3,S3,Class Weighting,5209,5210,4223,4224,986,987,4.278622,4.283976


In [14]:
checks = []

# --------------------------------------------
# Preprocessing
# --------------------------------------------

checks.append({
    "pemeriksaan":
        "Tidak ada missing pada training setelah imputasi",
    "lulus":
        preprocessing_audit[
            "training_missing_after"
        ].eq(0).all()
})

checks.append({
    "pemeriksaan":
        "Tidak ada missing pada validation setelah transformasi",
    "lulus":
        preprocessing_audit[
            "validation_missing_after"
        ].eq(0).all()
})

checks.append({
    "pemeriksaan":
        "Jumlah validation tidak berubah oleh preprocessing",
    "lulus":
        (
            preprocessing_audit["validation_rows_before"]
            ==
            preprocessing_audit["validation_rows_after"]
        ).all()
})

# --------------------------------------------
# S1
# --------------------------------------------

s1 = scenario_distribution[
    scenario_distribution["scenario"] == "S1"
]

checks.append({
    "pemeriksaan":
        "SMOTENC menghasilkan distribusi 1:1 pada seluruh training fold",
    "lulus":
        (
            s1["after_NON_ADHERENT"]
            ==
            s1["after_ADHERENT"]
        ).all()
})

# --------------------------------------------
# S2
# --------------------------------------------

s2 = scenario_distribution[
    scenario_distribution["scenario"] == "S2"
]

checks.append({
    "pemeriksaan":
        "RUS menghasilkan distribusi 1:1 pada seluruh training fold",
    "lulus":
        (
            s2["after_NON_ADHERENT"]
            ==
            s2["after_ADHERENT"]
        ).all()
})

# --------------------------------------------
# S3
# --------------------------------------------

s3 = scenario_distribution[
    scenario_distribution["scenario"] == "S3"
]

checks.append({
    "pemeriksaan":
        "Class weighting tidak mengubah jumlah observasi",
    "lulus":
        (
            s3["before_NON_ADHERENT"]
            ==
            s3["after_NON_ADHERENT"]
        ).all()
        and
        (
            s3["before_ADHERENT"]
            ==
            s3["after_ADHERENT"]
        ).all()
})

checks.append({
    "pemeriksaan":
        "Bobot ADHERENT lebih besar pada seluruh training fold",
    "lulus":
        (
            class_weight_summary["ADHERENT_weight"]
            >
            class_weight_summary["NON_ADHERENT_weight"]
        ).all()
})

pipeline_checks = pd.DataFrame(checks)

display(pipeline_checks)

pipeline_checks.to_csv(
    OUTPUT_DIR / "06_pipeline_checks.csv",
    index=False
)

,pemeriksaan,lulus
0,Tidak ada missing pada training setelah imputasi,True
1,Tidak ada missing pada validation setelah tran...,True
2,Jumlah validation tidak berubah oleh preproces...,True
3,SMOTENC menghasilkan distribusi 1:1 pada selur...,True
4,RUS menghasilkan distribusi 1:1 pada seluruh t...,True
5,Class weighting tidak mengubah jumlah observasi,True
6,Bobot ADHERENT lebih besar pada seluruh traini...,True


In [ ]:

# Verifikasi tambahan berdasarkan seluruh CSV hasil yang terbentuk.
# Catatan: pemeriksaan ini TIDAK membuktikan integrasi OneHotEncoder/fit estimator.
records=[]
expected = {(r,f) for r in (1,2,3) for f in (1,2,3,4,5)}
for name,frame in [
    ("preprocessing_audit",preprocessing_audit),
    ("class_weight_summary",class_weight_summary)]:
    keyset=set(zip(frame["repetition"],frame["fold"]))
    records.append({"pemeriksaan":f"{name}: 15 fold lengkap unik",
                    "lulus":len(frame)==15 and keyset==expected})
for s in ("S0","S1","S2","S3"):
    part=scenario_distribution[scenario_distribution["scenario"]==s]
    records.append({"pemeriksaan":f"Skenario {s}: 15 fold lengkap",
      "lulus":len(part)==15 and set(zip(part["repetition"],part["fold"]))==expected})
records.extend([
 {"pemeriksaan":"BENEFICIARY outer dan semua inner fold tidak overlap",
  "lulus":len(assignment_audit)==15 and assignment_audit["n_group_overlap"].eq(0).all()
  and assignment_audit["n_development_test_group_overlap"].eq(0).all()},
 {"pemeriksaan":"Jumlah dan label sebelum skenario sama untuk setiap fold",
  "lulus":all(g[["before_NON_ADHERENT","before_ADHERENT"]].nunique().eq(1).all()
      for _,g in scenario_distribution.groupby(["repetition","fold"]))},
 {"pemeriksaan":"SMOTENC tidak mengurangi kelas mayoritas dan mencapai 1:1",
  "lulus":((s1["after_NON_ADHERENT"]==s1["before_NON_ADHERENT"]) &
      (s1["after_ADHERENT"]==s1["after_NON_ADHERENT"])).all()},
 {"pemeriksaan":"RUS mempertahankan kelas minoritas dan mencapai 1:1",
  "lulus":((s2["after_ADHERENT"]==s2["before_ADHERENT"]) &
      (s2["after_NON_ADHERENT"]==s2["after_ADHERENT"])).all()},
])
extended_checks=pd.DataFrame(records)
extended_checks.to_csv(OUTPUT_DIR/"09_extended_pipeline_checks.csv",index=False)
display(extended_checks)
assert extended_checks["lulus"].all(), "Pemeriksaan tambahan gagal; audit output sebelum melanjutkan 4.4"
print("PASS: integritas partisi dan 15 fold. End-to-end model-fit belum diuji pada notebook ini.")


In [15]:
if pipeline_checks["lulus"].all():
    print("✓ Seluruh pemeriksaan preprocessing dan skenario berhasil.")
else:
    print("PERINGATAN: ada pemeriksaan yang belum sesuai.")

print("\nOutput tersimpan pada:")
print(OUTPUT_DIR)

✓ Seluruh pemeriksaan preprocessing dan skenario berhasil.

Output tersimpan pada:
/Users/riskyduha/Documents/Tesis/Prediction_Non_Adherent/output/02_preprocessing_scenario
